# 06 — Build SWOW SFT datasets (teacher warmup, small)

Metti il CSV in `data/raw/SWOW_EN.csv`.
Usiamo solo `cue` e `R1/R2/R3` (o raw).

In [1]:
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
swow_csv = RAW_DIR / 'SWOW_EN.csv'
swow_csv

WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw/SWOW_EN.csv')

In [2]:
import sys
sys.path.append(str((PROJECT_ROOT / 'src').resolve()))
from data.swow_to_sft import SWOWBuildConfig, load_swow_csv, to_pairs, expand_pairs, split_pairs, build_records
from utils.jsonl import write_jsonl

In [3]:
cfg = SWOWBuildConfig(seed=1234, max_train_examples=50_000, max_val_examples=2_000, max_test_examples=2_000)
cfg

SWOWBuildConfig(system_prompt='You are a helpful assistant. Given a cue word, output ONE likely associated word. Return ONLY the associated word.', seed=1234, val_frac=0.02, test_frac=0.02, max_train_examples=50000, max_val_examples=2000, max_test_examples=2000, min_len=1)

In [4]:
df = load_swow_csv(swow_csv)
print('Raw shape:', df.shape)
df.head()

Raw shape: (1356362, 18)


,Unnamed: 0,id,participantID,created_at,age,nativeLanguage,gender,education,city,country,section,cue,R1Raw,R2Raw,R3Raw,R1,R2,R3
0,1,1500428,130332,2018-01-07 04:29:38,61,United States,Ma,5.0,Pepperell,United States,seed,there,position,place,point,position,place,point
1,2,1500426,130332,2018-01-07 04:29:38,61,United States,Ma,5.0,Pepperell,United States,seed,true,honest,fact,indisputable,honest,fact,indisputable
2,3,1500424,130332,2018-01-07 04:29:38,61,United States,Ma,5.0,Pepperell,United States,seed,beat,drum,policeman,beatnik,drum,policeman,beatnik
3,4,1500438,130332,2018-01-07 04:29:38,61,United States,Ma,5.0,Pepperell,United States,seed,like,affection,simile,compare,affection,simile,compare
4,5,1500430,130332,2018-01-07 04:29:38,61,United States,Ma,5.0,Pepperell,United States,seed,telephone,receiver,hamdset,wires,receiver,handset,wires


In [5]:
pairs = to_pairs(df)
print('Pairs shape (pre-expand):', pairs.shape)
pairs.head()

Pairs shape (pre-expand): (4068965, 3)


,cue,response,count
0,there,position,1
1,there,place,1
2,there,point,1
3,true,honest,1
4,true,fact,1


In [6]:
pairs_exp = expand_pairs(pairs, seed=cfg.seed)
print('Expanded pairs:', pairs_exp.shape)
pairs_exp.head()

Expanded pairs: (4068965, 2)


,cue,response
0,daddy,gay
1,clock,hour
2,grade,color
3,probable,misunderstood
4,culture,mold


In [7]:
train_pairs, val_pairs, test_pairs = split_pairs(pairs_exp, cfg)
print('Split sizes:', len(train_pairs), len(val_pairs), len(test_pairs))

Split sizes: 3906207 81379 81379


In [8]:
train_rec = build_records(train_pairs, split='train', cfg=cfg, max_examples=cfg.max_train_examples)
val_rec = build_records(val_pairs, split='val', cfg=cfg, max_examples=cfg.max_val_examples)
test_rec = build_records(test_pairs, split='test', cfg=cfg, max_examples=cfg.max_test_examples)
print('Records:', len(train_rec), len(val_rec), len(test_rec))
train_rec[0]

Records: 50000 2000 2000


{'id': 'swow_train_0',
 'split': 'train',
 'source': 'swow',
 'cue': 'fresh',
 'messages': [{'role': 'system',
   'content': 'You are a helpful assistant. Given a cue word, output ONE likely associated word. Return ONLY the associated word.'},
  {'role': 'user', 'content': 'Cue: fresh\nAssociated word:'}],
 'assistant': 'smell'}

In [9]:
out_train = PROC_DIR / 'sft_teacher_swow_train.jsonl'
out_val   = PROC_DIR / 'sft_teacher_swow_val.jsonl'
out_test  = PROC_DIR / 'sft_teacher_swow_test.jsonl'
write_jsonl(train_rec, out_train)
write_jsonl(val_rec, out_val)
write_jsonl(test_rec, out_test)
print('Saved:', out_train)
print('Saved:', out_val)
print('Saved:', out_test)

Saved: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\sft_teacher_swow_train.jsonl
Saved: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\sft_teacher_swow_val.jsonl
Saved: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\sft_teacher_swow_test.jsonl
